# Detector de vehículos con cajas orientadas (OBB)

Este notebook es la guía reproducible del experimento. El entrenamiento vive en `edge_detection_experiment.py`: así el pipeline se puede ejecutar sin Jupyter, versionar, repetir y auditar.

El diseño separa clips completos en **train / validación / prueba**. Validación elige el checkpoint; las métricas finales se calculan una única vez sobre prueba.

## 1. Qué mide el experimento

En detección no existe una *accuracy* única útil: el fondo domina y la haría engañosa. Por ello se reportan Precision, Recall, F1, Macro AP@0.50 con IoU orientado, IoU medio de coincidencias, latencia, FPS, memoria, tamaño y parámetros. Son métricas apropiadas para la tabla de resultados de un paper.

Las imágenes se preparan con *letterboxing*: conservan proporción, tamaño y ángulo de las cajas OBB. El NMS es por clase y usa IoU orientado.

In [ ]:
from pathlib import Path
import csv
import sys

# Run from the repository root. If opened from this folder, adjust automatically.
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent.parent
SCRIPT = ROOT / 'main' / 'pytorch_implementation' / 'edge_detection_experiment.py'
OUTPUT_DIR = ROOT / 'reports' / 'paper_metrics' / 'experiments'
assert SCRIPT.exists(), f'No se encontró: {SCRIPT}'
print('Proyecto:', ROOT)
print('Pipeline:', SCRIPT)

## 2. Auditoría mínima de los cinco datasets

Cada tamaño usa su propio CSV e imágenes (`dataset_1000` ... `dataset_3000`), nunca `processed_300`. Esta celda comprueba que el número de filas e imágenes coincide antes de invertir tiempo en entrenamiento.

In [ ]:
DATASET_SIZES = [1000, 1500, 2000, 2500, 3000]
for size in DATASET_SIZES:
    folder = ROOT / 'data' / f'dataset_{size}'
    csv_path = folder / f'etiquetas_{size}.csv'
    image_count = len(list((folder / 'images').glob('*.jpg')))
    with csv_path.open(newline='') as handle:
        annotation_count = sum(1 for _ in csv.DictReader(handle))
    print(f'dataset_{size}: {annotation_count} anotaciones, {image_count} imágenes')

## 3. Configuración experimental

Para una prueba rápida, usa un solo dataset, una época y `MAX_TRAIN_BATCHES = 1`. Para resultados publicables, usa todos los tamaños, varias semillas y un número de épocas definido antes de mirar el test.

In [ ]:
EPOCHS = 8
BATCH_SIZE = 4
IMAGE_SIZE = 512
SEED = 42
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15
MAX_TRAIN_BATCHES = 0  # 0 = usar todos los batches

assert VALIDATION_RATIO + TEST_RATIO < 1
print('Split por clip:', 1 - VALIDATION_RATIO - TEST_RATIO, VALIDATION_RATIO, TEST_RATIO)

## 4. Ejecutar entrenamiento, selección y prueba

El script crea datasets independientes para entrenamiento y evaluación, conserva clips enteros en un solo split, guarda el mejor checkpoint según validación y después evalúa solamente en prueba. Las curvas y figuras se escriben en `reports/paper_metrics/experiments/`.

In [ ]:
import subprocess

command = [
    sys.executable, str(SCRIPT),
    '--dataset-sizes', *map(str, DATASET_SIZES),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--img-size', str(IMAGE_SIZE),
    '--val-ratio', str(VALIDATION_RATIO),
    '--test-ratio', str(TEST_RATIO),
    '--seed', str(SEED),
    '--max-train-batches', str(MAX_TRAIN_BATCHES),
    '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(command))
# Descomenta para iniciar el experimento.
# subprocess.run(command, cwd=ROOT, check=True)

## 5. Resultados para el paper

Después de entrenar, esta celda muestra la tabla final y los gráficos generados. Reporta siempre los campos de test, no los de validación, como resultados finales. Para conclusiones sólidas, repite cada condición con varias semillas y resume media ± desviación estándar.

In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage

summary_path = OUTPUT_DIR / 'experiment_summary.csv'
if summary_path.exists():
    results = pd.read_csv(summary_path)
    columns = ['dataset_size', 'train_images', 'val_images', 'test_images', 'macro_ap50_obb', 'precision_obb_iou50', 'recall_obb_iou50', 'f1_obb_iou50', 'mean_matched_iou_obb', 'fps_cpu']
    display(results[columns].round(4))
    display(DisplayImage(filename=str(OUTPUT_DIR / 'test_metric_comparison.png')))
else:
    print('Aún no hay resultados. Ejecuta la celda anterior.')

## 6. Limitaciones y siguientes pasos

Este es un baseline CPU compacto. Predice un objeto por celda: si dos centros caen en la misma celda, se conserva el objeto mayor. Antes de comparar modelos, registra esa tasa de colisiones. Para mejorar accuracy: usar múltiples anchors o un detector OBB moderno, augmentaciones geométricas que transformen correctamente las cajas, búsqueda de hiperparámetros solo en validación y repetición con varias semillas.